# Phase 1: Data Acquisition & Validation

## Maternal Care Provider Network Analysis

---

### Project Overview

This notebook loads and validates all datasets required for our OB-GYN provider network analysis. The goal is to map maternal care provider referral networks across the U.S., identify regions with strong vs. fragmented care coordination, and overlay maternal health outcomes to reveal where virtual care models could fill the biggest gaps.

### Datasets

| # | Dataset | File | Source | Study Period |
|---|---------|------|--------|-------------|
| 1 | CMS Physician Shared Patient Patterns | `pspp2015_60.csv` | [NBER](https://www.nber.org/research/data/physician-shared-patient-patterns-data) | 2015, 60-day window |
| 2 | NPPES NPI Bulk File | `npidata_pfile_20050523-20260208.csv` | [CMS](https://download.cms.gov/nppes/NPI_Files.html) | Current (Feb 2026) |
| 3 | NUCC Taxonomy Code Set | `nucc_taxonomy_151.csv` | [NUCC](https://www.nucc.org/) | Version 15.1 (7/1/2015) |
| 4a | CDC WONDER — Total Births by County | `cdc_wonder_2016_births_by_county.csv` | [CDC WONDER](https://wonder.cdc.gov/natality-expanded-current.html) | 2016 |
| 4b | CDC WONDER — Maternal Morbidity by County | `cdc_wonder_2016_maternal_morbidity_by_county.csv` | [CDC WONDER](https://wonder.cdc.gov/natality-expanded-current.html) | 2016 |

### Study Period Rationale

The CMS Shared Patient Patterns data (2009–2015) is the binding constraint. We use **2015** as the most recent and complete year. The **60-day window** captures referral-to-completion lag time — the realistic gap between when an OB-GYN refers a patient and when the specialist visit actually occurs. CDC WONDER data starts at **2016**, which aligns well since CDC natality statistics for a given year reflect births (and associated morbidity) from the prior reporting period.

### Assumptions & Limitations

- **NPPES Current vs. Historical:** A 2015-era NPPES snapshot is not freely available from CMS or NBER. We use the current NPPES bulk file (Feb 2026) under the assumption that OB-GYN taxonomy codes (207V) have not changed and that the majority of established practices have stable locations.
- **NUCC Version 15.1 (7/1/2015):** Selected to match the 2015 study period. OB-GYN taxonomy codes in the 207V family have been stable across versions.
- **Privacy Threshold:** The CMS shared patient file excludes provider pairs with fewer than 11 shared patients, meaning very low-volume referral relationships are not captured.
- **County Population Threshold:** CDC WONDER only reports county-level data for counties with populations >= 100,000 (per 2010 Census). Smaller counties are grouped as "Unidentified Counties" per state.

---

## Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from io import StringIO
from datetime import datetime

print(f"Notebook executed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")

Notebook executed: 2026-02-16 09:58:58
pandas version: 2.3.3
numpy version: 1.26.4


In [2]:
# -------------------------------------------------------------------
# FILE PATHS
# -------------------------------------------------------------------
# All paths are relative to the notebooks/ directory.
# Project structure:
#   maternal_network/
#   ├── notebooks/          <- you are here
#   ├── data/
#   │   ├── inputs/raw/
#   │   │   ├── cms_shared_patient/
#   │   │   ├── nppes/
#   │   │   ├── nucc_taxonomy/
#   │   │   └── cdc_wonder/
#   │   └── outputs/
# -------------------------------------------------------------------

RAW_DIR = Path("../data/inputs/raw")

CMS_FILE = RAW_DIR / "cms_shared_patient" / "pspp2015_60.csv"
NPPES_FILE = RAW_DIR / "nppes" / "npidata_pfile_20050523-20260208.csv"
NUCC_FILE = RAW_DIR / "nucc_taxonomy" / "nucc_taxonomy_151.csv"
CDC_BIRTHS_FILE = RAW_DIR / "cdc_wonder" / "cdc_wonder_2016_births_by_county.csv"
CDC_MORBIDITY_FILE = RAW_DIR / "cdc_wonder" / "cdc_wonder_2016_maternal_morbidity_by_county.csv"

# Verify all files exist
all_files = {
    "CMS Shared Patient Patterns": CMS_FILE,
    "NPPES NPI Bulk File": NPPES_FILE,
    "NUCC Taxonomy Code Set": NUCC_FILE,
    "CDC WONDER — Births by County": CDC_BIRTHS_FILE,
    "CDC WONDER — Maternal Morbidity": CDC_MORBIDITY_FILE,
}

print(f"{'Dataset':<40} {'Status':<10} {'Size':>12}")
print("-" * 65)
for name, fpath in all_files.items():
    if fpath.exists():
        size = fpath.stat().st_size
        if size > 1024**3:
            size_str = f"{size / (1024**3):.2f} GB"
        elif size > 1024**2:
            size_str = f"{size / (1024**2):.1f} MB"
        else:
            size_str = f"{size / 1024:.1f} KB"
        print(f"{name:<40} {'FOUND':<10} {size_str:>12}")
    else:
        print(f"{name:<40} {'MISSING':<10} {'--':>12}")

Dataset                                  Status             Size
-----------------------------------------------------------------
CMS Shared Patient Patterns              FOUND           2.83 GB
NPPES NPI Bulk File                      FOUND          10.44 GB
NUCC Taxonomy Code Set                   FOUND          425.0 KB
CDC WONDER — Births by County            FOUND           24.0 KB
CDC WONDER — Maternal Morbidity          FOUND           21.6 KB


---

## Dataset 1 — CMS Physician Shared Patient Patterns (2015, 60-Day Window)

**File:** `pspp2015_60.csv` (~2.9 GB)

**Description:** Each row represents a pair of providers who shared at least 11 Medicare patients within a rolling 60-day window during 2015. A shared patient event occurs when a beneficiary receives a billed service from Provider A and subsequently from Provider B within 60 days.

**Data Dictionary:**

| Column | Type | Description |
|--------|------|-------------|
| `npi1` | string | NPI of the initial/referring provider |
| `npi2` | string | NPI of the subsequent/referred-to provider |
| `paircount` | int | Total shared patient encounters in the 60-day window |
| `benecount` | int | Number of unique beneficiaries (patients) shared |
| `samedaycount` | int | Number of same-day shared encounters |
| `year` | int | Data year (2015) |
| `days` | int | Time window in days (60) |
| `begdate` | string | Window start date |
| `enddate` | string | Window end date |

In [3]:
# -------------------------------------------------------------------
# LOAD SAMPLE: CMS Shared Patient Patterns (first 10 rows)
# -------------------------------------------------------------------

cms_sample = pd.read_csv(
    CMS_FILE,
    nrows=10,
    dtype={"npi1": str, "npi2": str}
)

print("CMS Shared Patient Patterns — Sample (first 10 rows):")
print(f"Columns ({len(cms_sample.columns)}): {list(cms_sample.columns)}")
display(cms_sample)
print("\nData types:")
print(cms_sample.dtypes)

CMS Shared Patient Patterns — Sample (first 10 rows):
Columns (9): ['npi1', 'npi2', 'paircount', 'benecount', 'samedaycount', 'year', 'days', 'begdate', 'enddate']


,npi1,npi2,paircount,benecount,samedaycount,year,days,begdate,enddate
0,1000000004,1790775229,13,12,13,2015,60,1/1/2015,10/1/2015
1,1000026017,1598773715,69,23,29,2015,60,1/1/2015,10/1/2015
2,1000310429,1144645573,87,12,36,2015,60,1/1/2015,10/1/2015
3,1003000126,1003951625,200,62,28,2015,60,1/1/2015,10/1/2015
4,1003000126,1003975400,67,27,3,2015,60,1/1/2015,10/1/2015
5,1003000126,1013051119,40,19,0,2015,60,1/1/2015,10/1/2015
6,1003000126,1013902600,42,23,2,2015,60,1/1/2015,10/1/2015
7,1003000126,1023027109,48,17,7,2015,60,1/1/2015,10/1/2015
8,1003000126,1023029964,20,13,1,2015,60,1/1/2015,10/1/2015
9,1003000126,1033187000,29,11,1,2015,60,1/1/2015,10/1/2015



Data types:
npi1            object
npi2            object
paircount        int64
benecount        int64
samedaycount     int64
year             int64
days             int64
begdate         object
enddate         object
dtype: object


In [4]:
# -------------------------------------------------------------------
# VALIDATE: Confirm expected columns
# -------------------------------------------------------------------

CMS_EXPECTED_COLS = ["npi1", "npi2", "paircount", "benecount", "samedaycount",
                     "year", "days", "begdate", "enddate"]

missing_cols = [c for c in CMS_EXPECTED_COLS if c not in cms_sample.columns]
extra_cols = [c for c in cms_sample.columns if c not in CMS_EXPECTED_COLS]

if not missing_cols:
    print(f"PASS: All {len(CMS_EXPECTED_COLS)} expected columns found.")
else:
    print(f"FAIL: Missing columns: {missing_cols}")

if extra_cols:
    print(f"NOTE: Extra columns found: {extra_cols}")

PASS: All 9 expected columns found.


In [5]:
# -------------------------------------------------------------------
# VALIDATE: Row count (memory-efficient line count)
# -------------------------------------------------------------------

print("Counting total rows (this may take 1-2 minutes for a 2.9 GB file)...")
cms_row_count = sum(1 for _ in open(CMS_FILE)) - 1  # subtract header
print(f"Total data rows: {cms_row_count:,}")
print(f"File size: {CMS_FILE.stat().st_size / (1024**3):.2f} GB")

Counting total rows (this may take 1-2 minutes for a 2.9 GB file)...
Total data rows: 49,335,361
File size: 2.83 GB


In [6]:
# -------------------------------------------------------------------
# VALIDATE: Integrity checks
# -------------------------------------------------------------------

print("Integrity checks on sample:")
print(f"  NPI1 length (expect 10):          {cms_sample['npi1'].str.len().unique()}")
print(f"  NPI2 length (expect 10):          {cms_sample['npi2'].str.len().unique()}")
print(f"  Year values (expect 2015):        {cms_sample['year'].unique()}")
print(f"  Days values (expect 60):          {cms_sample['days'].unique()}")
print(f"  Min benecount (expect >= 11):     {cms_sample['benecount'].min()}")
print(f"  Null counts:")
for col in cms_sample.columns:
    n = cms_sample[col].isnull().sum()
    if n > 0:
        print(f"    {col}: {n}")
    else:
        print(f"    {col}: 0")

Integrity checks on sample:
  NPI1 length (expect 10):          [10]
  NPI2 length (expect 10):          [10]
  Year values (expect 2015):        [2015]
  Days values (expect 60):          [60]
  Min benecount (expect >= 11):     11
  Null counts:
    npi1: 0
    npi2: 0
    paircount: 0
    benecount: 0
    samedaycount: 0
    year: 0
    days: 0
    begdate: 0
    enddate: 0


---

## Dataset 2 — NPPES NPI Bulk File

**File:** `npidata_pfile_20050523-20260208.csv` (~11 GB, 300+ columns)

**Description:** The complete registry of all U.S. healthcare providers assigned a National Provider Identifier. We only load the columns needed for this project to avoid memory issues.

**Key columns for this project:**

| Column | Type | Description |
|--------|------|-------------|
| `NPI` | string | National Provider Identifier (join key to CMS data) |
| `Entity Type Code` | string | 1 = Individual, 2 = Organization |
| `Provider Last Name (Legal Name)` | string | Provider surname |
| `Provider First Name` | string | Provider first name |
| `Healthcare Provider Taxonomy Code_1` through `_15` | string | Up to 15 specialty taxonomy codes |
| `Provider Business Practice Location Address City Name` | string | Practice city |
| `Provider Business Practice Location Address State Name` | string | Practice state |
| `Provider Business Practice Location Address Postal Code` | string | Practice ZIP code |

In [7]:
# -------------------------------------------------------------------
# DEFINE: Columns to load from NPPES
# -------------------------------------------------------------------

NPPES_USE_COLS = [
    "NPI",
    "Entity Type Code",
    "Provider Last Name (Legal Name)",
    "Provider First Name",
    "Provider Business Practice Location Address City Name",
    "Provider Business Practice Location Address State Name",
    "Provider Business Practice Location Address Postal Code",
] + [f"Healthcare Provider Taxonomy Code_{i}" for i in range(1, 16)]

print(f"Columns to load: {len(NPPES_USE_COLS)}")
for col in NPPES_USE_COLS:
    print(f"  {col}")

Columns to load: 22
  NPI
  Entity Type Code
  Provider Last Name (Legal Name)
  Provider First Name
  Provider Business Practice Location Address City Name
  Provider Business Practice Location Address State Name
  Provider Business Practice Location Address Postal Code
  Healthcare Provider Taxonomy Code_1
  Healthcare Provider Taxonomy Code_2
  Healthcare Provider Taxonomy Code_3
  Healthcare Provider Taxonomy Code_4
  Healthcare Provider Taxonomy Code_5
  Healthcare Provider Taxonomy Code_6
  Healthcare Provider Taxonomy Code_7
  Healthcare Provider Taxonomy Code_8
  Healthcare Provider Taxonomy Code_9
  Healthcare Provider Taxonomy Code_10
  Healthcare Provider Taxonomy Code_11
  Healthcare Provider Taxonomy Code_12
  Healthcare Provider Taxonomy Code_13
  Healthcare Provider Taxonomy Code_14
  Healthcare Provider Taxonomy Code_15


In [8]:
# -------------------------------------------------------------------
# VALIDATE: Confirm our columns exist in the file header
# -------------------------------------------------------------------

nppes_header = pd.read_csv(NPPES_FILE, nrows=0, dtype=str, low_memory=False)
print(f"Total columns in NPPES file: {len(nppes_header.columns)}")

missing = [c for c in NPPES_USE_COLS if c not in nppes_header.columns]
if not missing:
    print(f"PASS: All {len(NPPES_USE_COLS)} required columns found.")
else:
    print(f"FAIL: Missing columns: {missing}")

Total columns in NPPES file: 330
PASS: All 22 required columns found.


In [9]:
# -------------------------------------------------------------------
# LOAD SAMPLE: NPPES (first 10 rows, selected columns only)
# -------------------------------------------------------------------

nppes_sample = pd.read_csv(
    NPPES_FILE,
    usecols=NPPES_USE_COLS,
    nrows=10,
    dtype=str,
    low_memory=False
)

print(f"NPPES — Sample (first 10 rows, {len(nppes_sample.columns)} columns):")
display(nppes_sample)
print("\nData types:")
print(nppes_sample.dtypes)

NPPES — Sample (first 10 rows, 22 columns):


,NPI,Entity Type Code,Provider Last Name (Legal Name),Provider First Name,Provider Business Practice Location Address City Name,Provider Business Practice Location Address State Name,Provider Business Practice Location Address Postal Code,Healthcare Provider Taxonomy Code_1,Healthcare Provider Taxonomy Code_2,Healthcare Provider Taxonomy Code_3,...,Healthcare Provider Taxonomy Code_6,Healthcare Provider Taxonomy Code_7,Healthcare Provider Taxonomy Code_8,Healthcare Provider Taxonomy Code_9,Healthcare Provider Taxonomy Code_10,Healthcare Provider Taxonomy Code_11,Healthcare Provider Taxonomy Code_12,Healthcare Provider Taxonomy Code_13,Healthcare Provider Taxonomy Code_14,Healthcare Provider Taxonomy Code_15
0,1679576722,1,WIEBE,DAVID,KEARNEY,NE,688472944,207X00000X,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1588667638,1,PILCHER,WILLIAM,JACKSONVILLE,FL,322044736,207RC0000X,207RC0000X,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1497758544,2,NaN,NaN,FAYETTEVILLE,NC,283044552,251G00000X,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1306849450,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1215930367,1,GRESSOT,LAURENT,HOUSTON,TX,770901243,174400000X,207RH0003X,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,1023011178,2,NaN,NaN,NAPA,CA,945594515,251G00000X,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1932102084,1,ADUSUMILLI,RAVI,TOLEDO,OH,436151753,207RC0000X,207RC0000X,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,1841293990,1,WORTSMAN,SUSAN,NEW YORK,NY,100102547,231H00000X,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,1750384806,1,BISBEE,ROBERT,LUBBOCK,TX,794151148,207R00000X,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,1669475711,1,SUNG,BIN,FULSHEAR,TX,774411548,208000000X,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Data types:
NPI                                                        object
Entity Type Code                                           object
Provider Last Name (Legal Name)                            object
Provider First Name                                        object
Provider Business Practice Location Address City Name      object
Provider Business Practice Location Address State Name     object
Provider Business Practice Location Address Postal Code    object
Healthcare Provider Taxonomy Code_1                        object
Healthcare Provider Taxonomy Code_2                        object
Healthcare Provider Taxonomy Code_3                        object
Healthcare Provider Taxonomy Code_4                        object
Healthcare Provider Taxonomy Code_5                        object
Healthcare Provider Taxonomy Code_6                        object
Healthcare Provider Taxonomy Code_7                        object
Healthcare Provider Taxonomy Code_8                        obje

In [10]:
# -------------------------------------------------------------------
# VALIDATE: Integrity checks
# -------------------------------------------------------------------

print("Integrity checks on NPPES sample:")
print(f"  NPI length (expect 10):       {nppes_sample['NPI'].str.len().unique()}")
print(f"  Entity Type codes (1 or 2):   {nppes_sample['Entity Type Code'].unique()}")
print(f"  States sample:                {nppes_sample['Provider Business Practice Location Address State Name'].unique()}")
print(f"  Taxonomy Code_1 sample:       {nppes_sample['Healthcare Provider Taxonomy Code_1'].unique()}")
print(f"  File size:                    {NPPES_FILE.stat().st_size / (1024**3):.2f} GB")

Integrity checks on NPPES sample:
  NPI length (expect 10):       [10]
  Entity Type codes (1 or 2):   ['1' '2' nan]
  States sample:                ['NE' 'FL' 'NC' nan 'TX' 'CA' 'OH' 'NY']
  Taxonomy Code_1 sample:       ['207X00000X' '207RC0000X' '251G00000X' nan '174400000X' '231H00000X'
 '207R00000X' '208000000X']
  File size:                    10.44 GB


---

## Dataset 3 — NUCC Taxonomy Code Set (Version 15.1)

**File:** `nucc_taxonomy_151.csv`

**Description:** The National Uniform Claim Committee (NUCC) taxonomy code set maps each provider specialty to a standardized code. We use the 207V family to identify OB-GYN providers.

**Data Dictionary:**

| Column | Type | Description |
|--------|------|-------------|
| `Code` | string | Taxonomy code (e.g., `207V00000X`) |
| `Type` | string | Broad provider type (e.g., "Allopathic & Osteopathic Physicians") |
| `Classification` | string | Specialty classification (e.g., "Obstetrics & Gynecology") |
| `Specialization` | string | Sub-specialty (e.g., "Maternal & Fetal Medicine") |
| `Definition` | string | Description of the specialty |
| `Notes` | string | Additional notes |

In [11]:
# -------------------------------------------------------------------
# LOAD: NUCC Taxonomy Code Set
# -------------------------------------------------------------------

nucc_df = pd.read_csv(NUCC_FILE, dtype=str)

print(f"NUCC Taxonomy Code Set loaded.")
print(f"Total taxonomy codes: {len(nucc_df):,}")
print(f"Columns: {list(nucc_df.columns)}")
display(nucc_df.head())

NUCC Taxonomy Code Set loaded.
Total taxonomy codes: 838
Columns: ['Code', 'Type', 'Classification', 'Specialization', 'Definition', 'Notes']


,Code,Type,Classification,Specialization,Definition,Notes
0,101Y00000X,Behavioral Health & Social Service Providers,Counselor,NaN,A provider who is trained and educated in the ...,Sources: Abridged from definitions provided by...
1,101YA0400X,Behavioral Health & Social Service Providers,Counselor,Addiction (Substance Use Disorder),Definition to come...,NaN
2,101YM0800X,Behavioral Health & Social Service Providers,Counselor,Mental Health,Definition to come...,NaN
3,101YP1600X,Behavioral Health & Social Service Providers,Counselor,Pastoral,Definition to come...,NaN
4,101YP2500X,Behavioral Health & Social Service Providers,Counselor,Professional,Definition to come...,NaN


In [12]:
# -------------------------------------------------------------------
# VALIDATE: Filter to OB-GYN codes (207V family)
# -------------------------------------------------------------------

obgyn_codes = nucc_df[nucc_df["Code"].str.startswith("207V", na=False)].copy()

print(f"OB-GYN taxonomy codes found (207V family): {len(obgyn_codes)}")
print()
display(obgyn_codes[["Code", "Classification", "Specialization"]])

OB-GYN taxonomy codes found (207V family): 10



,Code,Classification,Specialization
282,207V00000X,Obstetrics & Gynecology,NaN
283,207VB0002X,Obstetrics & Gynecology,Obesity Medicine
284,207VC0200X,Obstetrics & Gynecology,Critical Care Medicine
285,207VE0102X,Obstetrics & Gynecology,Reproductive Endocrinology
286,207VF0040X,Obstetrics & Gynecology,Female Pelvic Medicine and Reconstructive Surgery
287,207VG0400X,Obstetrics & Gynecology,Gynecology
288,207VH0002X,Obstetrics & Gynecology,Hospice and Palliative Medicine
289,207VM0101X,Obstetrics & Gynecology,Maternal & Fetal Medicine
290,207VX0000X,Obstetrics & Gynecology,Obstetrics
291,207VX0201X,Obstetrics & Gynecology,Gynecologic Oncology


---

## Dataset 4 — CDC WONDER Natality Expanded (2016)

**Files:**
- `cdc_wonder_2016_births_by_county.csv` — Total births per county (denominator)
- `cdc_wonder_2016_maternal_morbidity_by_county.csv` — Births with at least one maternal morbidity event per county (numerator)

**Description:** County-level birth counts from CDC WONDER, exported as tab-delimited files with metadata rows appended at the bottom (below a `"---"` separator). We use these two files to compute a **maternal morbidity rate** per county: morbidity births / total births.

**Maternal morbidity indicators included in "at least one checked":**
- Maternal Transfusion
- Third or Fourth Degree Perineal Laceration
- Ruptured Uterus
- Unplanned Hysterectomy
- Admission to Intensive Care Unit

**Data Dictionary (both files share the same structure):**

| Column | Description |
|--------|-------------|
| `Notes` | Metadata (usually empty for data rows) |
| `County of Residence` | County and state name (e.g., "Baldwin County, AL") |
| `County of Residence Code` | 5-digit FIPS code (e.g., "01003") |
| `Births` | Count of births (total or morbidity, depending on file) |

**Note:** Counties with population < 100,000 are grouped as "Unidentified Counties" per state. Suppressed values may appear where counts are too low.

In [13]:
# -------------------------------------------------------------------
# HELPER: Load CDC WONDER export
# -------------------------------------------------------------------
# CDC WONDER appends metadata rows below a "---" separator.
# This function reads only the data rows above that line.
# Delimiter is auto-detected (tab or comma) since CDC WONDER
# exports may use either format despite the .csv extension.
# -------------------------------------------------------------------

def load_cdc_wonder(filepath):
    """Load a CDC WONDER export, stopping at the metadata separator."""
    lines = []
    with open(filepath, 'r') as f:
        for line in f:
            if line.startswith('"---'):
                break
            lines.append(line)
    
    # Detect delimiter from header line
    header = lines[0]
    delimiter = '\t' if '\t' in header else ','
    
    return pd.read_csv(StringIO(''.join(lines)), sep=delimiter, dtype=str)

In [14]:
# -------------------------------------------------------------------
# LOAD & VALIDATE: CDC WONDER — Total Births by County (2016)
# -------------------------------------------------------------------

cdc_births = load_cdc_wonder(CDC_BIRTHS_FILE)

print(f"CDC WONDER — Total Births by County (2016):")
print(f"Rows: {len(cdc_births):,}")
print(f"Columns: {list(cdc_births.columns)}")
display(cdc_births.head(10))

# Check for non-numeric birth counts (suppressed or missing values)
non_numeric = cdc_births[pd.to_numeric(cdc_births['Births'], errors='coerce').isna()]
print(f"\nNon-numeric Births values: {len(non_numeric)} rows")
if len(non_numeric) > 0:
    print("(These are likely 'Suppressed' or 'Not Applicable' — will be handled in Phase 2)")
    display(non_numeric.head())

CDC WONDER — Total Births by County (2016):
Rows: 627
Columns: ['Notes', 'County of Residence', 'County of Residence Code', 'Births']


,Notes,County of Residence,County of Residence Code,Births
0,NaN,"Baldwin County, AL",01003,2247
1,NaN,"Calhoun County, AL",01015,1357
2,NaN,"Etowah County, AL",01055,1214
3,NaN,"Houston County, AL",01069,1333
4,NaN,"Jefferson County, AL",01073,8655
5,NaN,"Lee County, AL",01081,1898
6,NaN,"Madison County, AL",01089,4228
7,NaN,"Mobile County, AL",01097,5504
8,NaN,"Montgomery County, AL",01101,3139
9,NaN,"Morgan County, AL",01103,1470



Non-numeric Births values: 0 rows


In [15]:
# -------------------------------------------------------------------
# LOAD & VALIDATE: CDC WONDER — Maternal Morbidity by County (2016)
# -------------------------------------------------------------------

cdc_morbidity = load_cdc_wonder(CDC_MORBIDITY_FILE)

print(f"CDC WONDER — Maternal Morbidity by County (2016):")
print(f"Rows: {len(cdc_morbidity):,}")
print(f"Columns: {list(cdc_morbidity.columns)}")
display(cdc_morbidity.head(10))

# Check for non-numeric birth counts
non_numeric = cdc_morbidity[pd.to_numeric(cdc_morbidity['Births'], errors='coerce').isna()]
print(f"\nNon-numeric Births values: {len(non_numeric)} rows")
if len(non_numeric) > 0:
    print("(These are likely 'Suppressed' — will be handled in Phase 2)")
    display(non_numeric.head())

CDC WONDER — Maternal Morbidity by County (2016):
Rows: 578
Columns: ['Notes', 'County of Residence', 'County of Residence Code', 'Births']


,Notes,County of Residence,County of Residence Code,Births
0,NaN,"Baldwin County, AL",01003,43
1,NaN,"Calhoun County, AL",01015,16
2,NaN,"Etowah County, AL",01055,11
3,NaN,"Jefferson County, AL",01073,50
4,NaN,"Lee County, AL",01081,26
5,NaN,"Madison County, AL",01089,82
6,NaN,"Mobile County, AL",01097,76
7,NaN,"Montgomery County, AL",01101,27
8,NaN,"Morgan County, AL",01103,28
9,NaN,"Shelby County, AL",01117,17



Non-numeric Births values: 0 rows


In [16]:
# -------------------------------------------------------------------
# VALIDATE: Compare county coverage between the two CDC files
# -------------------------------------------------------------------

births_counties = set(cdc_births["County of Residence Code"].dropna())
morbidity_counties = set(cdc_morbidity["County of Residence Code"].dropna())

print(f"Counties in births file:      {len(births_counties):,}")
print(f"Counties in morbidity file:   {len(morbidity_counties):,}")
print(f"Counties in both:             {len(births_counties & morbidity_counties):,}")
print(f"In births only:               {len(births_counties - morbidity_counties):,}")
print(f"In morbidity only:            {len(morbidity_counties - births_counties):,}")

if births_counties - morbidity_counties:
    print(f"\nNOTE: Counties in births but not in morbidity may have had zero")
    print(f"morbidity events or were suppressed due to small counts.")
    print(f"These will be treated as zero morbidity in Phase 2.")

Counties in births file:      626
Counties in morbidity file:   577
Counties in both:             577
In births only:               49
In morbidity only:            0

NOTE: Counties in births but not in morbidity may have had zero
morbidity events or were suppressed due to small counts.
These will be treated as zero morbidity in Phase 2.


---

## Data Acquisition Summary

In [17]:
# -------------------------------------------------------------------
# SUMMARY: Final status report
# -------------------------------------------------------------------

print("=" * 75)
print("PHASE 1 — DATA ACQUISITION & VALIDATION SUMMARY")
print("=" * 75)

summary = [
    ("CMS Shared Patient Patterns", CMS_FILE,
     f"{cms_row_count:,} rows"),
    ("NPPES NPI Bulk File", NPPES_FILE,
     f"{len(nppes_header.columns)} columns"),
    ("NUCC Taxonomy Code Set (v15.1)", NUCC_FILE,
     f"{len(nucc_df):,} codes ({len(obgyn_codes)} OB-GYN)"),
    ("CDC WONDER — Births", CDC_BIRTHS_FILE,
     f"{len(cdc_births):,} counties"),
    ("CDC WONDER — Morbidity", CDC_MORBIDITY_FILE,
     f"{len(cdc_morbidity):,} counties"),
]

print(f"\n{'Dataset':<40} {'Size':>12} {'Detail':>25}")
print("-" * 75)

all_ready = True
for name, fpath, detail in summary:
    if fpath.exists():
        size = fpath.stat().st_size
        if size > 1024**3:
            size_str = f"{size / (1024**3):.2f} GB"
        elif size > 1024**2:
            size_str = f"{size / (1024**2):.1f} MB"
        else:
            size_str = f"{size / 1024:.1f} KB"
        print(f"{name:<40} {size_str:>12} {detail:>25}")
    else:
        print(f"{name:<40} {'MISSING':>12} {'--':>25}")
        all_ready = False

print("-" * 75)
if all_ready:
    print("\nAll datasets loaded and validated.")
    print("Ready to proceed to Phase 2: Data Preparation (02_data_preparation.ipynb)")
else:
    print("\nSome datasets are missing. Complete downloads before proceeding.")

PHASE 1 — DATA ACQUISITION & VALIDATION SUMMARY

Dataset                                          Size                    Detail
---------------------------------------------------------------------------
CMS Shared Patient Patterns                   2.83 GB           49,335,361 rows
NPPES NPI Bulk File                          10.44 GB               330 columns
NUCC Taxonomy Code Set (v15.1)               425.0 KB     838 codes (10 OB-GYN)
CDC WONDER — Births                           24.0 KB              627 counties
CDC WONDER — Morbidity                        21.6 KB              578 counties
---------------------------------------------------------------------------

All datasets loaded and validated.
Ready to proceed to Phase 2: Data Preparation (02_data_preparation.ipynb)


---

## Next Steps

Proceed to **`02_data_preparation.ipynb`**:
1. Filter NPPES to OB-GYN providers using 207V taxonomy codes
2. Filter the CMS shared patient edge list to only OB-GYN provider pairs
3. Merge provider details (sub-specialty, practice location) onto both sides of each pair
4. Compute county-level maternal morbidity rates from the CDC WONDER exports
5. Export clean, analysis-ready datasets to `data/outputs/`